# RLSSM Simulators: Advanced Tutorial

This is the **advanced** companion to the
[RLSSM beginner tutorial](rlssm_tutorial.ipynb). It assumes you already know the
basics: an RLSSM couples a **learning process**, a **decision process (SSM)**, and a
**task environment**, simulated in an interleaved trial-by-trial loop, and you build
one with `rl.ModelConfig` + `rl.Simulator`.

Here we open up the parts the beginner tutorial deliberately skipped:

1. **Response labels vs. learning actions** — and how to control the mapping.
2. **Swapping components** — dual-learning-rate learners, Gaussian rewards, the
   `TaskConfig` shorthand, and writing your *own* learning process / task environment.
3. **Participant-wise parameters** — the full rules for per-participant `theta`.
4. **Posterior predictive simulation** (`mode="ppc"`).
5. **Data-validation failure modes** — what a *bad* dataset looks like.
6. **Inference internals** — compiled models and how the **HSSM bridge** works
   under the hood.

> **A note on optional dependencies.** Everything here runs with just
> `ssm-simulators`. One cell demonstrates the differentiable **JAX** backend and is
> written to degrade gracefully if the optional `[jax]` extra is not installed. The
> **HSSM** handoff is shown as *illustrative* code (not executed), since HSSM is a
> separate package.

## Setup

We reuse one set of parameter values (`theta`) and the `2AB_RW_Angle` preset as a
baseline throughout.

In [1]:
import ssms.rl as rl

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Baseline model + parameters reused across the tutorial.
base_config = rl.preset.get("2AB_RW_Angle")
theta = {
    "rl_alpha": 0.3,
    "scaler": 2.0,
    "a": 1.5,
    "z": 0.5,
    "t": 0.3,
    "theta": 0.2,
}

# A small valid dataset we reuse for PPC, validation, and the bridge sections.
data = rl.Simulator(base_config).simulate(
    theta=theta, n_trials=60, n_participants=5, random_state=42
)
data.head()

,participant_id,trial_id,rt,response,feedback
0,0,0,2.999996,1,0.0
1,0,1,1.506524,-1,0.0
2,0,2,1.080149,-1,0.0
3,0,3,1.355910,-1,1.0
4,0,4,1.051847,-1,0.0


## 1. Response labels vs. learning actions

This distinction is the single most important "advanced" idea, because it is where
the SSM world and the RL world meet.

- The **decision process** (the SSM) emits a **response label**. For a two-choice
  angle/DDM model these are `-1` and `1`.
- The **learning process** and the **task environment** think in **zero-based action
  indices**: `0, 1, ... , n_arms - 1`.

So on every trial the simulator must translate the SSM's response label into an action
index before sampling a reward and updating Q-values. Turn on `include_action=True` to
see both side by side.

In [2]:
labeled = rl.ModelConfig(
    model_name="labeled",
    description="expose both response label and action index",
    decision_process="angle",
    learning_process=rl.learning.RescorlaWagnerDeltaRule(),
    task_environment=rl.env.Bandit.bernoulli(
        probabilities=[0.7, 0.3], response_labels=[-1, 1]
    ),
    include_action=True,
)

d = rl.Simulator(labeled).simulate(theta=theta, n_trials=6, n_participants=1, random_state=0)
print("response -> action mapping:", labeled.response_to_action)
d[["trial_id", "response", "action", "feedback"]]

response -> action mapping: {-1: 0, 1: 1}


,trial_id,response,action,feedback
0,0,1,1,0.0
1,1,-1,0,0.0
2,2,1,1,0.0
3,3,1,1,0.0
4,4,-1,0,0.0
5,5,-1,0,1.0


By default `response_mapping="auto"`, which pairs response labels with action indices
**in the order the labels were given**: `response_labels=[-1, 1]` becomes
`{-1: 0, 1: 1}`. You can also pass an explicit dictionary — for example to *reverse*
the meaning of the labels:

In [3]:
reversed_map = rl.ModelConfig(
    model_name="reversed",
    description="custom, reversed response mapping",
    decision_process="angle",
    learning_process=rl.learning.RescorlaWagnerDeltaRule(),
    task_environment=rl.env.Bandit.bernoulli(
        probabilities=[0.7, 0.3], response_labels=[-1, 1]
    ),
    response_mapping={-1: 1, 1: 0},   # flip which arm each label updates
)
reversed_map.response_to_action

{-1: 1, 1: 0}

A mapping is validated when the config is built: its keys must cover **exactly** the
response labels, and its values must be **unique** and exactly the indices
`0 .. n_arms - 1`. Anything else raises a clear error.

## 2. Swapping in different components

The three pillars are independent — you can replace any one of them.

### A dual-learning-rate learner

`RescorlaWagnerDualAlphaRule` uses a **separate** learning rate for positive vs.
negative prediction errors (`rl_alpha` for `reward >= Q`, `rl_alpha_neg` otherwise),
a common way to model optimism/pessimism. It adds one parameter to `theta`.

In [4]:
dual = rl.learning.RescorlaWagnerDualAlphaRule()
print("free params:", dual.free_params)

dual_config = rl.ModelConfig(
    model_name="dual_alpha",
    description="asymmetric Rescorla-Wagner",
    decision_process="angle",
    learning_process=dual,
    task_environment=rl.env.Bandit.bernoulli(probabilities=[0.7, 0.3], response_labels=[-1, 1]),
)
theta_dual = {**theta, "rl_alpha_neg": 0.1}  # learn less from negative outcomes
rl.Simulator(dual_config).simulate(theta=theta_dual, n_trials=5, n_participants=1, random_state=0).head()

free params: ['rl_alpha', 'rl_alpha_neg', 'scaler']


,participant_id,trial_id,rt,response,feedback
0,0,0,0.752191,1,0.0
1,0,1,3.762216,-1,0.0
2,0,2,2.644401,1,0.0
3,0,3,1.402577,1,0.0
4,0,4,1.131324,-1,0.0


### Continuous (Gaussian) rewards

Bandits are not limited to 0/1 feedback. `Bandit.gaussian(...)` draws continuous
rewards from per-arm normal distributions.

In [5]:
gaussian_config = rl.ModelConfig(
    model_name="gaussian_reward",
    description="continuous Gaussian rewards",
    decision_process="angle",
    learning_process=rl.learning.RescorlaWagnerDeltaRule(),
    task_environment=rl.env.Bandit.gaussian(
        means=[1.0, 0.0], sds=[0.5, 0.5], response_labels=[-1, 1]
    ),
)
g = rl.Simulator(gaussian_config).simulate(theta=theta, n_trials=5, n_participants=1, random_state=0)
g[["trial_id", "response", "feedback"]]   # feedback is now continuous

,trial_id,response,feedback
0,0,1,-0.447973
1,1,-1,1.367978
2,2,-1,1.426691
3,3,-1,1.080474
4,4,-1,1.402828


### The `TaskConfig` shorthand

For the built-in bandits you can skip constructing the environment object and pass a
`TaskConfig` instead; the config builds the environment for you.

In [6]:
shorthand_config = rl.ModelConfig(
    model_name="shorthand",
    description="task built from a TaskConfig",
    decision_process="angle",
    learning_process=rl.learning.RescorlaWagnerDeltaRule(),
    task_environment=rl.env.TaskConfig(
        task="bandit", reward="gaussian", means=[1.0, 0.0], sds=[0.5, 0.5],
        response_labels=[-1, 1],
    ),
)
type(shorthand_config.task_environment).__name__   # resolved to a Bandit

'Bandit'

### Writing your own task environment

A task environment is any object implementing the `rl.env.TaskEnvironment` protocol:
the properties `n_arms`, `response_labels`, `extra_fields`, and the methods
`reset(rng)`, `sample_reward(action, trial_idx)`, and `get_extra_data(trial_idx)`.

Here is a **reversal bandit** — the better arm flips halfway through — which is not a
built-in. Note how `get_extra_data` adds a custom `best_arm` column to the output.

In [7]:
class ReversalBandit:
    '''Two-armed Bernoulli bandit whose best arm flips at `reversal_at`.'''

    def __init__(self, p_high=0.8, response_labels=(-1, 1), reversal_at=50):
        self.p_high = p_high
        self._labels = list(response_labels)
        self.reversal_at = reversal_at
        self._rng = None

    @property
    def n_arms(self):
        return 2

    @property
    def response_labels(self):
        return list(self._labels)

    @property
    def extra_fields(self):
        return ["best_arm"]

    def reset(self, rng=None):
        self._rng = rng or np.random.default_rng()

    def _probs(self, trial_idx):
        hi = self.p_high
        return [hi, 1 - hi] if trial_idx < self.reversal_at else [1 - hi, hi]

    def sample_reward(self, action, trial_idx):
        if self._rng is None:
            raise RuntimeError("call reset() before sample_reward()")
        return float(self._rng.random() < self._probs(trial_idx)[action])

    def get_extra_data(self, trial_idx):
        return {"best_arm": float(int(np.argmax(self._probs(trial_idx))))}


# It satisfies the protocol:
print("is a TaskEnvironment:", isinstance(ReversalBandit(), rl.env.TaskEnvironment))

reversal_config = rl.ModelConfig(
    model_name="reversal",
    description="custom reversal-bandit environment",
    decision_process="angle",
    learning_process=rl.learning.RescorlaWagnerDeltaRule(),
    task_environment=ReversalBandit(reversal_at=50),
    response_mapping={-1: 0, 1: 1},
)
rev = rl.Simulator(reversal_config).simulate(theta=theta, n_trials=100, n_participants=20, random_state=1)
print("best_arm before reversal:", rev.loc[rev.trial_id < 50, "best_arm"].mean())
print("best_arm after reversal :", rev.loc[rev.trial_id >= 50, "best_arm"].mean())
rev[["trial_id", "response", "feedback", "best_arm"]].head()

is a TaskEnvironment: True


best_arm before reversal: 0.0
best_arm after reversal : 1.0


,trial_id,response,feedback,best_arm
0,0,1,1.0,0.0
1,1,1,0.0,0.0
2,2,-1,1.0,0.0
3,3,-1,0.0,0.0
4,4,-1,0.0,0.0


### Writing your own learning process

Likewise, a learning process implements the `rl.learning.LearningProcess` interface.
The simulator drives it through its **explicit-state** methods — `init_state()`,
`compute_python(state, params)` (computes the SSM parameter *before* the update), and
`update_python(state, action, reward, params)` (returns the next state) — plus the
declarative properties that describe its parameters. Here is a compact,
NumPy-only Rescorla–Wagner learner written from scratch.

In [8]:
class SimpleRW:
    '''Minimal Rescorla-Wagner learner (NumPy only). drift = (Q1 - Q0) * scaler.'''

    # --- declarative description of the process ---
    @property
    def computed_params(self):
        return ["v"]                       # the SSM parameter it produces

    @property
    def free_params(self):
        return ["rl_alpha", "scaler"]      # parameters it needs from theta

    @property
    def param_bounds(self):
        return {"rl_alpha": (0.0, 1.0), "scaler": (0.001, 10.0)}

    @property
    def default_params(self):
        return {"rl_alpha": 0.2, "scaler": 2.0}

    @property
    def available_backends(self):
        return ("python",)                 # NumPy only (no JAX gradient path)

    @property
    def supports_gradient(self):
        return False

    # --- explicit-state simulation interface ---
    def init_state(self):
        return {"q": np.array([0.5, 0.5])}

    def compute_python(self, state, params):
        # drift computed from current Q-values, BEFORE this trial's update
        return {"v": float((state["q"][1] - state["q"][0]) * params["scaler"])}

    def update_python(self, state, action, reward, params):
        q = state["q"].copy()
        q[action] += params["rl_alpha"] * (reward - q[action])
        return {"q": q}

    def reset(self, **kwargs):
        self._state = self.init_state()


custom_lp_config = rl.ModelConfig(
    model_name="custom_lp",
    description="hand-written learning process",
    decision_process="angle",
    learning_process=SimpleRW(),
    task_environment=rl.env.Bandit.bernoulli(probabilities=[0.7, 0.3], response_labels=[-1, 1]),
)
print("resolved backend:", custom_lp_config.resolved_learning_backend,
      "| gradient:", custom_lp_config.resolved_gradient)
rl.Simulator(custom_lp_config).simulate(theta=theta, n_trials=5, n_participants=1, random_state=0).head()

resolved backend: python | gradient: unavailable


,participant_id,trial_id,rt,response,feedback
0,0,0,0.752191,1,0.0
1,0,1,3.282251,-1,0.0
2,0,2,2.644401,1,0.0
3,0,3,1.415578,1,0.0
4,0,4,1.058325,-1,0.0


## 3. Participant-wise parameters (the full rules)

The beginner tutorial showed that a parameter can be a list (one value per
participant). The complete rules are:

- **Scalars** are shared by (tiled across) all participants.
- **Lists/arrays** are per-participant; all participant-wise lists must have the
  **same length**.
- If `n_participants` is omitted, it is **inferred** from that length.
- If you *also* pass `n_participants`, it must **match** the inferred length.

You can freely mix scalar and list parameters:

In [9]:
mixed = {
    **theta,
    "rl_alpha": [0.1, 0.3, 0.6],   # three participants
    "a": [1.2, 1.5, 1.8],          # also per-participant; same length
}
out = rl.Simulator(base_config).simulate(theta=mixed, n_trials=30, random_state=0)
print("inferred participants:", sorted(out["participant_id"].unique()))

inferred participants: [np.int64(0), np.int64(1), np.int64(2)]


The guardrails produce explicit errors. Two common mistakes:

In [10]:
sim = rl.Simulator(base_config)

# (1) participant-wise lists of different lengths
try:
    sim.simulate(theta={**theta, "rl_alpha": [0.1, 0.3], "a": [1.2, 1.5, 1.8]}, n_trials=10)
except ValueError as e:
    print("length mismatch ->", e)

# (2) explicit n_participants disagreeing with the list length
try:
    sim.simulate(theta={**theta, "rl_alpha": [0.1, 0.3, 0.6]}, n_participants=5, n_trials=10)
except ValueError as e:
    print("count conflict  ->", e)

length mismatch -> participant-wise theta values must all have the same length. Got lengths by param: {'rl_alpha': 2, 'a': 3}.
count conflict  -> n_participants=5 does not match the participant-wise theta length 3.


## 4. Posterior predictive simulation (`mode="ppc"`)

`mode="generative"` (the default) simulates a fresh experiment from scratch.
`mode="ppc"` instead performs **observed-history-conditioned** simulation, the
workhorse of posterior predictive checks: for every trial of an *observed* dataset it

- computes the learning state from the **observed** history up to that trial,
- simulates a **new** `rt` and `response`,
- copies the **observed** outcome (`feedback`) into the output, and
- updates the learning state using the **observed** response and outcome.

So the design and outcomes are held fixed to the empirical data, while RTs and choices
are re-drawn from the model — exactly what you want when checking whether a fitted
model reproduces behavior.

In [11]:
observed = rl.Simulator(base_config).simulate(
    theta=theta, n_trials=60, n_participants=5, random_state=1
)
ppc = rl.Simulator(base_config).simulate(
    theta=theta, mode="ppc", observed_data=observed, random_state=2
)

print("feedback copied from observed:", bool((ppc["feedback"].values == observed["feedback"].values).all()))
print("responses re-simulated       :", bool((ppc["response"].values != observed["response"].values).any()))
ppc.head()

feedback copied from observed: True
responses re-simulated       : True


,participant_id,trial_id,rt,response,feedback
0,0,0,2.659885,-1,1.0
1,0,1,1.236296,1,0.0
2,0,2,1.273693,-1,1.0
3,0,3,3.062061,-1,0.0
4,0,4,3.228644,-1,0.0


## 5. Data validation — failure modes

The beginner tutorial showed a *passing* `validate_data` report. In practice the
report earns its keep when something is **wrong**. It checks required columns, a
balanced and trial-ordered panel, valid response labels, and missing values — and the
messages include repair hints.

Here we feed it a broken frame whose outcome column has the wrong name:

In [12]:
broken = data.rename(columns={"feedback": "reward"})   # config expects 'feedback'

report = base_config.validate_data(broken)
report.print()

# `raise_for_errors()` turns the report into an exception you can guard on:
try:
    report.raise_for_errors()
except Exception as e:
    print("\nraised ->", type(e).__name__)

RLSSM data validation report:
  [ERROR] missing_column: Required column 'feedback' is missing.
         hint: Rename column 'reward' to 'feedback', or set ModelConfig(outcome_field='reward').

raised -> ValueError


## 6. Inference internals: compiled models and the HSSM bridge

Simulation only needs the learning process to run *forward*. Inference (in HSSM) needs
it to be **differentiable**, so it can compute gradients of the likelihood. That is
what **compiling** a model surfaces.

In [13]:
compiled = base_config.compile(backend="auto")
print("learning backend:", compiled.learning_backend)
print("gradient support:", compiled.gradient)

learning backend: jax
gradient support: available


With `backend="auto"` the model uses JAX when the optional `[jax]` extra is installed
(enabling gradients) and falls back to NumPy otherwise. The next cell asks explicitly
for the JAX backend and degrades gracefully if JAX is unavailable:

In [14]:
try:
    jax_compiled = base_config.compile(backend="jax")
    print("JAX backend OK — gradient:", jax_compiled.gradient)
except Exception as e:
    print("JAX backend unavailable:", e)
    print("Install with:  pip install 'ssm-simulators[jax]'")

JAX backend OK — gradient: available


### The participant-wise computed-parameter function

The piece HSSM actually consumes is a function that replays a participant's history and
returns the **computed SSM parameters** for every trial. `CompiledModel` derives the
input layout from the config and builds that function for you. Below we use the NumPy
backend so it runs anywhere.

In [15]:
py_config = rl.ModelConfig(
    model_name="2AB_RW_Angle_py",
    description="python-backend copy for a portable demo",
    decision_process="angle",
    learning_process=rl.learning.RescorlaWagnerDeltaRule(),
    task_environment=rl.env.Bandit.bernoulli(probabilities=[0.7, 0.3], response_labels=[-1, 1]),
    learning_backend="python",
    gradient="unavailable",
)
compiled_py = py_config.compile(backend="python")

fields = compiled_py.participant_input_fields()
print("input fields (column order):", fields)

compute_v = compiled_py.compile_participant_fn(output="dict")

# One participant's history: columns in `fields` order = [rl_alpha, scaler, response, feedback]
history = np.array([
    [0.3, 2.0, -1, 0.0],
    [0.3, 2.0, -1, 1.0],
    [0.3, 2.0,  1, 0.0],
    [0.3, 2.0, -1, 1.0],
])
print("computed drift per trial:", np.round(compute_v(history)["v"], 3))

input fields (column order): ['rl_alpha', 'scaler', 'response', 'feedback']
computed drift per trial: [ 0.    0.3  -0.09 -0.39]


The first trial's drift is `0` (the Q-values start equal) and then evolves as feedback
arrives — and crucially the drift for a trial is computed *before* that trial's update,
matching HSSM's inference-side scan order.

### How HSSM consumes this

HSSM provides a bridge factory that wires all of the above together. Conceptually,
`RLSSMConfig.from_ssms_model(...)`:

1. resolves the `ssms.rl` model (`resolve_model`),
2. compiles it with the JAX backend and **requires** `gradient == "available"`,
3. wraps `participant_input_fields()` + `compile_participant_fn(output="dict")` into
   HSSM's annotated computed-parameter functions,
4. builds the decision-process (ONNX) likelihood itself, and
5. validates the data via `validate_data(...).raise_for_errors()`.

The simulator side exposes exactly the structural metadata HSSM needs:

In [16]:
bridge = base_config.to_hssm_config_dict()
print("model_name      :", bridge["model_name"])
print("decision_process:", bridge["decision_process"])
print("list_params     :", bridge["list_params"])
bridge["participant_contract"]

model_name      : 2AB_RW_Angle
decision_process: angle
list_params     : ['rl_alpha', 'scaler', 'a', 'z', 't', 'theta']


{'trial_params': ['rl_alpha', 'scaler'],
 'computed_outputs': ['v'],
 'response_field': 'response',
 'outcome_field': 'feedback',
 'input_fields': ['rl_alpha', 'scaler', 'response', 'feedback']}

On the HSSM side you do not assemble any of this by hand. The following is
**illustrative** — it requires HSSM (a separate package) and is not executed here:

```python
import hssm

# Build a ready-to-fit HSSM model straight from the ssms.rl model.
hssm_config = hssm.rl.RLSSMConfig.from_ssms_model("2AB_RW_Angle")  # or a ModelConfig
model = hssm.RLSSM(data=data, model_config=hssm_config)

idata = model.sample()   # standard PyMC sampling
```

See the [`ssms.rl` API reference](../api/rlssm.md) for the full inference workflow.

## 7. Summary

You now have the full simulator-side toolkit:

- **Response labels vs. actions**, and `response_mapping` (`"auto"` or explicit).
- **Interchangeable components**: dual-alpha learners, Gaussian rewards, the
  `TaskConfig` shorthand, and your own `TaskEnvironment` / `LearningProcess`.
- **Participant-wise `theta`** with its full inference/validation rules.
- **`mode="ppc"`** for observed-history-conditioned posterior predictive simulation.
- **Validation failure modes** and how to read the repair hints.
- **Compiled models** and a conceptual map of the **HSSM bridge**.

Looking ahead, richer task environments (such as built-in reversal bandits) are planned
as the framework grows. For the foundations, revisit the
[beginner tutorial](rlssm_tutorial.ipynb); for the full reference, see the
[`ssms.rl` API page](../api/rlssm.md).